# BhramGuard Text Model Training

Train a phishing classifier for message or email text from `backend/ml/datasets/texts.csv`.

This notebook starts with a classical NLP baseline: TF-IDF word features plus handcrafted phishing-language signals. It saves model artifacts into `backend/ml/models`.

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[2]

DATASET_PATH = PROJECT_ROOT / "backend" / "ml" / "datasets" / "texts.csv"
TEST_DATASET_PATH = PROJECT_ROOT / "backend" / "ml" / "datasets" / "phishing_legitimate_dataset.csv"
MODEL_DIR = PROJECT_ROOT / "backend" / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)
print("Optional test dataset:", TEST_DATASET_PATH)
print("Models:", MODEL_DIR)

Project root: c:\Users\Bidur\Desktop\BhramGuard-
Dataset: c:\Users\Bidur\Desktop\BhramGuard-\backend\ml\datasets\texts.csv
Optional test dataset: c:\Users\Bidur\Desktop\BhramGuard-\backend\ml\datasets\phishing_legitimate_dataset.csv
Models: c:\Users\Bidur\Desktop\BhramGuard-\backend\ml\models


In [4]:
import re

import joblib
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

In [5]:
df_text = pd.read_csv(DATASET_PATH)

print("Shape:", df_text.shape)
print("Columns:", df_text.columns.tolist())
df_text.head()

Shape: (20137, 3)
Columns: ['text', 'label', 'text_length']


,text,label,text_length
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",0,1030
1,the other side of * galicismos * * galicismo *...,0,479
2,re : equistar deal tickets are you still avail...,0,1245
3,\r\nHello I am your hot lil horny toy.\r\n ...,1,688
4,software at incredibly low prices ( 86 % lower...,1,441


In [6]:
def infer_columns(df):
    label_candidates = ["label", "Label", "class", "target", "is_phishing"]
    text_candidates = ["text", "message", "email", "body", "content"]

    label_col = next((col for col in label_candidates if col in df.columns), None)
    if label_col is None:
        raise ValueError(f"No label column found. Columns: {df.columns.tolist()}")

    text_col = next((col for col in text_candidates if col in df.columns and col != label_col), None)
    if text_col is None:
        object_cols = [col for col in df.columns if col != label_col and df[col].dtype == "object"]
        if not object_cols:
            raise ValueError(f"No text column found. Columns: {df.columns.tolist()}")
        text_col = object_cols[0]

    return text_col, label_col

TEXT_COL, LABEL_COL = infer_columns(df_text)
print("Text column:", TEXT_COL)
print("Label column:", LABEL_COL)
print(df_text[LABEL_COL].value_counts(dropna=False))

Text column: text
Label column: label
label
0    12465
1     7672
Name: count, dtype: int64


In [7]:
df_text = df_text[[TEXT_COL, LABEL_COL]].dropna()
df_text[TEXT_COL] = df_text[TEXT_COL].astype(str)
df_text[LABEL_COL] = df_text[LABEL_COL].astype(int)
df_text = df_text.drop_duplicates(subset=[TEXT_COL])

print("Cleaned shape:", df_text.shape)
print(df_text[LABEL_COL].value_counts(normalize=True))

Cleaned shape: (20137, 2)
label
0    0.61901
1    0.38099
Name: proportion, dtype: float64


In [8]:
def extract_text_features(value):
    text = "" if pd.isna(value) else str(value)
    lowered = text.lower()
    words = text.split()

    urgency_words = ["hurry", "urgent", "immediately", "act now", "limited time", "expire", "act fast"]
    reward_words = ["free", "win", "winner", "prize", "discount", "offer", "guarantee", "cash"]
    threat_words = ["suspend", "verify", "confirm", "account", "password", "security", "unauthorized"]
    authority_words = ["bank", "paypal", "microsoft", "google", "admin", "support", "team", "irs", "tax"]
    caps_words = [word for word in words if word.isupper() and len(word) > 1]

    return {
        "char_length": len(text),
        "word_count": len(words),
        "urgency_count": sum(1 for word in urgency_words if word in lowered),
        "reward_count": sum(1 for word in reward_words if word in lowered),
        "threat_count": sum(1 for word in threat_words if word in lowered),
        "authority_count": sum(1 for word in authority_words if word in lowered),
        "caps_ratio": len(caps_words) / len(words) if words else 0,
        "exclamation_count": text.count("!"),
        "question_count": text.count("?"),
        "link_count": len(re.findall(r"https?://|www\.", lowered)),
        "email_count": len(re.findall(r"[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}", text)),
        "money_count": len(re.findall(r"[$]\s?\d+|\d+\s?(usd|dollars|rs|npr)", lowered)),
        "elongation_count": len(re.findall(r"([a-zA-Z])\1{3,}", text)),
    }

manual_features = pd.DataFrame(df_text[TEXT_COL].apply(extract_text_features).tolist())
manual_features.describe().T

,count,mean,std,min,25%,50%,75%,max
char_length,20137.0,2512.576203,120116.596284,2.0,281.0,751.0,1693.000000,17036692.0
word_count,20137.0,486.752048,24866.874567,0.0,51.0,136.0,319.000000,3527576.0
urgency_count,20137.0,0.073943,0.291321,0.0,0.0,0.0,0.000000,6.0
reward_count,20137.0,0.669117,0.970675,0.0,0.0,0.0,1.000000,8.0
threat_count,20137.0,0.182450,0.500183,0.0,0.0,0.0,0.000000,6.0
authority_count,20137.0,0.413666,0.794678,0.0,0.0,0.0,1.000000,7.0
caps_ratio,20137.0,0.018148,0.063308,0.0,0.0,0.0,0.007576,1.0
exclamation_count,20137.0,2.221036,101.448814,0.0,0.0,0.0,1.000000,14368.0
question_count,20137.0,1.741223,52.636878,0.0,0.0,0.0,1.000000,6096.0
link_count,20137.0,0.989075,32.862073,0.0,0.0,0.0,0.000000,4637.0


In [9]:
X_text_train, X_text_test, X_manual_train, X_manual_test, y_train, y_test = train_test_split(
    df_text[TEXT_COL],
    manual_features,
    df_text[LABEL_COL],
    test_size=0.2,
    random_state=42,
    stratify=df_text[LABEL_COL],
)

tfidf = TfidfVectorizer(
    max_features=5000,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    stop_words="english",
)

X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

X_train = hstack([X_train_tfidf, csr_matrix(X_manual_train.values)])
X_test = hstack([X_test_tfidf, csr_matrix(X_manual_test.values)])

print("Train matrix:", X_train.shape)
print("Test matrix:", X_test.shape)

Train matrix: (16109, 5013)
Test matrix: (4028, 5013)


In [10]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "linear_svc": LinearSVC(class_weight="balanced", random_state=42),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", n_jobs=-1),
}

results = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    auc = roc_auc_score(y_test, probabilities) if probabilities is not None else None

    print("\n===", model_name, "===")
    print(classification_report(y_test, predictions))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))
    if auc is not None:
        print("ROC-AUC:", auc)

    report = classification_report(y_test, predictions, output_dict=True)
    results[model_name] = {"model": model, "auc": auc, "f1": report["weighted avg"]["f1-score"]}

best_model_name = max(results, key=lambda name: (results[name]["auc"] or 0, results[name]["f1"]))
best_model = results[best_model_name]["model"]
print("Best model:", best_model_name)

c:\Users\Bidur\Desktop\BhramGuard-\backend\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



=== logistic_regression ===
              precision    recall  f1-score   support

           0       0.92      0.92      0.92      2493
           1       0.87      0.87      0.87      1535

    accuracy                           0.90      4028
   macro avg       0.90      0.90      0.90      4028
weighted avg       0.90      0.90      0.90      4028

Confusion matrix:
[[2297  196]
 [ 200 1335]]
ROC-AUC: 0.949827987420151

=== linear_svc ===
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      2493
           1       0.91      0.93      0.92      1535

    accuracy                           0.94      4028
   macro avg       0.93      0.93      0.93      4028
weighted avg       0.94      0.94      0.94      4028

Confusion matrix:
[[2345  148]
 [ 109 1426]]

=== gradient_boosting ===
              precision    recall  f1-score   support

           0       0.91      0.96      0.93      2493
           1       0.93      0.84      0.88 

In [11]:
if TEST_DATASET_PATH.exists():
    df_external = pd.read_csv(TEST_DATASET_PATH).dropna()
    external_text_col, external_label_col = infer_columns(df_external)
    df_external[external_text_col] = df_external[external_text_col].astype(str)
    df_external[external_label_col] = df_external[external_label_col].astype(int)

    external_manual = pd.DataFrame(df_external[external_text_col].apply(extract_text_features).tolist())
    external_tfidf = tfidf.transform(df_external[external_text_col])
    external_X = hstack([external_tfidf, csr_matrix(external_manual.values)])
    external_y = df_external[external_label_col]
    external_pred = best_model.predict(external_X)

    print("External test dataset:", TEST_DATASET_PATH)
    print(classification_report(external_y, external_pred))
    print("Confusion matrix:")
    print(confusion_matrix(external_y, external_pred))

External test dataset: c:\Users\Bidur\Desktop\BhramGuard-\backend\ml\datasets\phishing_legitimate_dataset.csv
              precision    recall  f1-score   support

           0       0.71      0.88      0.79        25
           1       0.84      0.64      0.73        25

    accuracy                           0.76        50
   macro avg       0.78      0.76      0.76        50
weighted avg       0.78      0.76      0.76        50

Confusion matrix:
[[22  3]
 [ 9 16]]


In [12]:
artifact = {
    "model": best_model,
    "vectorizer": tfidf,
    "manual_feature_columns": manual_features.columns.tolist(),
    "text_column": TEXT_COL,
    "label_column": LABEL_COL,
    "best_model_name": best_model_name,
}

joblib.dump(artifact, MODEL_DIR / "text_phishing_model.pkl")
print("Saved:", MODEL_DIR / "text_phishing_model.pkl")

Saved: c:\Users\Bidur\Desktop\BhramGuard-\backend\ml\models\text_phishing_model.pkl
